# OCR Export Notebook

`load_pdfs_as_documents()`가 만든 최종 하이브리드 텍스트를 `easyocr.txt` 폴더에 저장합니다.

- 내장 텍스트가 충분히 좋으면 `native`
- 품질이 낮은 페이지는 `ocr`
- 둘을 합친 최종본을 `.txt`로 저장


In [ ]:
from pathlib import Path
import os
import sys
from collections import Counter

# EasyOCR/torch import 전에 GPU 2번만 보이도록 제한합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "easyocr.txt" else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from chroma.data_loader import load_pdfs_as_documents

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "easyocr.txt"
OVERWRITE = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
docs = load_pdfs_as_documents(str(RAW_DATA_PATH))
print("문서 수:", len(docs))
print("타입 분포:", dict(Counter(doc.metadata.get("type", "unknown") for doc in docs)))
print("품질 상태 분포:", dict(Counter(doc.metadata.get("quality_status", "unknown") for doc in docs)))


In [ ]:
def make_output_path(doc, raw_data_path, output_dir):
    file_path = doc.metadata.get("file_path")
    source_name = doc.metadata.get("source", "document.pdf")

    if file_path:
        try:
            relative_path = Path(file_path).resolve().relative_to(Path(raw_data_path).resolve())
            return Path(output_dir) / relative_path.with_suffix(".txt")
        except Exception:
            pass

    company = doc.metadata.get("card_company", "unknown")
    return Path(output_dir) / company / (Path(source_name).stem + ".txt")


def export_docs_to_txt(docs, raw_data_path, output_dir, overwrite=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    skipped = []

    for doc in docs:
        text = (doc.page_content or "").strip()
        if not text:
            skipped.append({"file": doc.metadata.get("source", "document.pdf"), "reason": "empty_text"})
            continue

        output_path = make_output_path(doc, raw_data_path, output_dir)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists() and not overwrite:
            skipped.append({"file": output_path.name, "reason": "exists"})
            continue

        output_path.write_text(text, encoding="utf-8")
        saved.append(
            {
                "file": output_path.name,
                "type": doc.metadata.get("type"),
                "quality_status": doc.metadata.get("quality_status"),
                "ocr_pages": doc.metadata.get("ocr_pages", 0),
            }
        )

    return saved, skipped


In [ ]:
saved, skipped = export_docs_to_txt(docs, RAW_DATA_PATH, OUTPUT_DIR, overwrite=OVERWRITE)
print("저장 완료:", len(saved))
print("건너뜀:", len(skipped))
saved[:10], skipped[:10]


In [ ]:
sample_files = sorted(OUTPUT_DIR.glob("*.txt"))[:5]
for path in sample_files:
    print("=" * 100)
    print(path.name)
    print(path.read_text(encoding="utf-8")[:500])
    print()
